### 1. import and read 

In [18]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
events = duckdb.sql('SELECT * FROM "../dataset/events.csv";')
inventory_items = duckdb.sql('SELECT * FROM "../dataset/inventory_items.csv";')
orders =duckdb.sql("SELECT * FROM read_csv('../dataset/orders.csv', types={'gender': 'VARCHAR'})")
order_items = duckdb.sql('SELECT * FROM "../dataset/order_items.csv"')
users = duckdb.sql('SELECT * FROM "../dataset/users.csv"')

### 2. Data Cleaning + EDA

#### 2.1 events

In [20]:
#=====================================================
# 1.events
#=====================================================
clean_events =duckdb.sql("""
SELECT
    id,
    COALESCE(user_id,0) AS user_id,
    sequence_number,
    session_id,
    created_at,
    ip_address,
    CASE WHEN
        city = 'null' THEN 'Unknown' ELSE
        city
    END AS city,
    state,
    postal_code,
    browser,
    traffic_source,
    uri,
    event_type
FROM events
""")

**EDA**

#### 2.2 inventory_items

In [21]:
#=====================================================
# 2.inventory_items
#=====================================================
clean_inventory_items= duckdb.sql("""
SELECT
    id,
    product_id,
    created_at,
    sold_at,
    cost,
    product_category,
    COALESCE(product_name,'Unknown') AS product_name,
    COALESCE(product_brand,'Unknown') AS product_brand,
    product_retail_price,
    product_department,
    product_sku,
    product_distribution_center_id
FROM
    inventory_items
""")

#### 2.3 orders

In [22]:
#=====================================================
# 3. orders
#=====================================================
clean_orders= duckdb.sql("""
SELECT
    order_id,
    user_id,
    status,
    gender,
    TRY_CAST(created_at AS timestamp) AS created_at,
    TRY_CAST(returned_at AS timestamp) AS returned_at,
    TRY_CAST(shipped_at AS timestamp) AS shipped_at,
    TRY_CAST(delivered_at AS timestamp) AS delivered_at,
    num_of_item
FROM
    orders
""")

#### 2.4 order_items

**Issue Identified:** 
- During EDA, logical chronological validation (created_at > shipped_at | delivered_at) revealed 35,889 rows of dirty data in the foundational order items table that severely violate standard time sequencing.

**Diagnostic Analysis:** 
- Through time-gap investigation (with some inverted discrepancies exceeding 40 hours), the hypothesis of unaligned timezone offsets across cross-border business systems was ruled out. The root cause was determined to be a critical flaw in the data generation/entry logic of the upstream source system.

**Data Cleaning Strategy:** 
- Because our downstream Mart_Logistics_Experience requires calculating actual fulfillment duration, retaining these anomalies would yield "negative" logistics fulfillment times, completely undermining the foundation of our causal inference models. Consequently, these records were strictly dropped during the ELT transformation phase.

In [23]:
#=====================================================
# 4. order_items
#=====================================================
clean_order_items= duckdb.sql("""
SELECT
    id,
    order_id,
    user_id,
    inventory_item_id,
    status,
    TRY_CAST(created_at AS timestamp) AS created_at,
    TRY_CAST(shipped_at AS timestamp) AS shipped_at,
    TRY_CAST(delivered_at AS timestamp) AS delivered_at,
    TRY_CAST(returned_at AS timestamp) AS returned_at,
    sale_price
FROM order_items
WHERE
    created_at > returned_at or
    created_at > shipped_at or
    created_at > delivered_at
""")

#### 2.5 users

In [24]:
#=====================================================
# 5. users
#=====================================================
clean_users = duckdb.sql("""
SELECT
    id,
    first_name,
    last_name,
    COALESCE(email,'Unknown') AS email,
    COALESCE(age,null) AS age,
    COALESCE(gender,'Unknown') AS gender,
    state,
    COALESCE(street_address,'Unknown') AS street_address,
    postal_code,
    COALESCE(NULLIF(LOWER(city), 'null'), 'Unknown') AS city,
    country,
    latitude,
    longitude,
    traffic_source,
    COALESCE(created_at,null) AS created_at,
    user_geom
FROM users
""")

### 4. Craete OBTs 
- **Frontend Traffic OBT** (clean_users + clean_events)
- **Backend Sales OBT** (clean_order_items + clean_orders + clean_inventory_items + clean_users)

In [25]:
#==========================================================================================
# 1. Frontend Traffic OBT
#==========================================================================================
Frontend_Traffic_OBT =duckdb.sql("""
SELECT
    e.id AS event_id,
    e.user_id AS user_id,
    session_id,
    u.created_at AS user_created_at,
    e.created_at AS event_created_at,
    event_type,
    sequence_number,
    browser,
    u.traffic_source AS user_traffic_source,
    e.traffic_source AS event_traffic_source,
    age,
    gender,
    u.state AS user_home_state,
    e.state AS event_ip_state,
    u.city AS user_home_city,
    e.city AS event_ip_city,
    u.postal_code AS user_postal_code,
    latitude,
    longitude,
    uri
FROM clean_events AS e
LEFT JOIN clean_users AS u ON e.user_id = u.id
ORDER BY user_id
""")

#==========================================================================================
# 2. Backend Sales OBT
#==========================================================================================
Backend_Sales_OBT= duckdb.sql("""
WITH t1 AS(
    SELECT
        ii.id AS inventory_item_id,
        product_id,
        it.id AS order_items_id,
        product_distribution_center_id,
        ii.created_at AS inventory_items_created_at,
        sold_at,
        cost,
        product_category,
        product_name,
        product_brand,
        product_retail_price,
        product_department,
        product_sku,
        order_id,
        user_id,
        status AS items_status,
        it.created_at AS items_created_at,
        shipped_at AS items_shipped_at,
        delivered_at AS items_delivered_at,
        returned_at AS items_returned_at,
        sale_price
    FROM
        clean_inventory_items AS ii
    LEFT JOIN
        clean_order_items AS it ON ii.id = it.inventory_item_id
    )
SELECT
    t1.order_id AS order_id,
    order_items_id,
    inventory_item_id,
    product_id,
    t1.user_id AS user_id,
    product_distribution_center_id,
    o.created_at AS order_created_at,
    returned_at AS order_returned_at,
    shipped_at AS order_shipped_at,
    delivered_at AS order_delivered_at,
    items_created_at,
    items_returned_at,
    items_shipped_at,
    items_delivered_at,
    inventory_items_created_at,
    sold_at,
    u.created_at AS user_created_at,
    items_status,
    status AS orders_status,
    cost,
    product_retail_price,
    num_of_item,
    sale_price,
    age,
    u.gender,
    product_brand,
    product_department,
    product_category,
    product_name,
    traffic_source,
    state,
    postal_code,
    country,
    city,
    latitude,
    longitude,
    product_sku
FROM
    t1
LEFT JOIN
    clean_orders AS o ON t1.order_id = o.order_id
LEFT JOIN
    clean_users AS u ON t1.user_id = u.id
""")


### 5. Create Data Marks

##### 5.1 Mart_Financial_Summary

In [26]:
# Financial_Summary_Structure
# 1. Volume Metrics
# 2. Financial Metrics
# 3. Health Ratios
Mart_Financial_Summary=duckdb.sql("""
WITH Base_Metrics AS (
    SELECT
        CONCAT(YEAR(order_created_at), '-Q', QUARTER(order_created_at)) AS year_quarter,
        CONCAT('Q', QUARTER(order_created_at)) AS quarter,
        YEAR(order_created_at) AS year,

        COUNT(DISTINCT user_id) AS active_buyers,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(order_items_id) AS total_items_sold,

        SUM(CASE WHEN items_status = 'Returned' THEN 1 ELSE 0 END) AS returned_items_count,
        SUM(CASE WHEN items_status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_items_count,

        SUM(sale_price) AS GMV,
        SUM(CASE WHEN items_status = 'Returned' THEN sale_price ELSE 0 END) AS return_amount,
        SUM(CASE WHEN items_status = 'Cancelled' THEN sale_price ELSE 0 END) AS cancelled_amount,
        
        SUM(CASE WHEN items_status NOT IN ('Returned', 'Cancelled') THEN sale_price ELSE 0 END) AS net_revenue,
        SUM(CASE WHEN items_status NOT IN ('Returned', 'Cancelled') THEN cost ELSE 0 END) AS COGS

    FROM Backend_Sales_OBT
    WHERE user_id IS NOT NULL
    GROUP BY YEAR(order_created_at), QUARTER(order_created_at)
)

SELECT
    year_quarter,
    quarter,
    year,
    active_buyers,
    total_orders,
    total_items_sold,

    ROUND(return_amount, 2) AS return_amount,
    ROUND(cancelled_amount, 2) AS cancelled_amount,
    ROUND(GMV, 2) AS GMV,
    ROUND(net_revenue, 2) AS net_revenue,
    ROUND(COGS, 2) AS COGS,
    ROUND(net_revenue - COGS, 2) AS gross_profit,

    ROUND(100.0 * (net_revenue - COGS) / NULLIF(net_revenue, 0), 2) AS gross_margin_pct,
    ROUND(GMV / NULLIF(total_orders, 0), 2) AS AOV,
    ROUND(net_revenue / NULLIF(active_buyers, 0), 2) AS ARPU,
    ROUND(100.0 * returned_items_count / NULLIF(total_items_sold, 0), 2) AS return_rate_pct,
    ROUND(100.0 * cancelled_items_count / NULLIF(total_items_sold, 0), 2) AS cancellation_rate_pct

FROM Base_Metrics
ORDER BY Year, Quarter
""")


##### 5.2 Mart_RFM_Segmentation

In [27]:
Mart_RFM_Segmentation=duckdb.sql("""
WITH t AS(
    SELECT
        MAX(order_created_at) AS global_max_date
    FROM Backend_Sales_OBT
),t2 AS(
    SELECT
        CONCAT(YEAR(order_created_at), '-Q', QUARTER(order_created_at)) AS Year_Quarter,
        user_id,
        DATEDIFF('day',MAX(order_created_at), (SELECT global_max_date FROM t)) AS recency,
        COUNT(DISTINCT order_id) AS frequency,
        ROUND(SUM(sale_price),2) AS monetary
    FROM Backend_Sales_OBT
    WHERE
        items_status NOT IN ('Cancelled','Returned') AND 
        user_id IS NOT NULL AND 
        order_created_at >= (SELECT global_max_date - INTERVAL 1 YEAR FROM t)
    GROUP BY
        user_id,YEAR(order_created_at),QUARTER(order_created_at)
)
SELECT
    user_id,
    recency,
    frequency,
    monetary,
    NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
    NTILE(5) OVER (ORDER BY recency ASC) AS f_score,
    NTILE(5) OVER (ORDER BY monetary ASC) AS m_score,
    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
        WHEN r_score = 1 AND f_score >= 4 AND m_score >= 4 THEN 'Cant Lose Them' 
        WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Loyalists'
        WHEN r_score >= 4 AND f_score = 1 THEN 'New Customers'
        WHEN r_score >= 3 AND f_score <= 3 THEN 'Potential Loyalists' 
        WHEN r_score IN (2, 3) AND f_score IN (2, 3) THEN 'Need Attention'
        WHEN r_score <= 2 AND f_score >= 2 AND m_score >= 2 THEN 'At Risk'
        WHEN r_score <= 2 AND f_score >= 4 AND m_score <= 2 THEN 'Lost Bargain Hunters'
        WHEN r_score <= 2 AND f_score <= 2 AND m_score <= 2 THEN 'Hibernating / Lost'
        ELSE 'Others' 
    END AS rfm_segment
FROM t2
ORDER BY
    r_score DESC,f_score DESC,m_score DESC
""")

##### 5.3 Mart_Logistics_Experience

In [28]:
Mart_Logistics_Experience =duckdb.sql("""
SELECT
    user_id,
    order_id,
    COUNT(DISTINCT order_items_id) AS total_items,
    COUNT(DISTINCT product_distribution_center_id) AS total_dc,
    DATEDIFF('hour',MIN(order_created_at),MAX(items_delivered_at)) AS total_fulfillment_time_hours,
    MAX(CASE WHEN orders_status = 'Returned' THEN 1 ELSE 0 END) AS return_flag
FROM
    Backend_Sales_OBT
WHERE
    user_id IS NOT NULL AND items_delivered_at IS NOT NULL 
GROUP BY
    order_id,user_id
ORDER BY user_id 
""")

##### 5.4 Mart_Experimentation_Base

In [29]:
duckdb.sql("""
WITH frontend_agg AS (
    SELECT 
        user_id,
        ANY_VALUE(user_traffic_source) AS experiment_group,
        COUNT(event_id) AS total_page_view
    FROM Frontend_Traffic_OBT
    GROUP BY user_id
),
backend_agg AS (
    SELECT
        user_id,
        SUM(sale_price) AS total_spend
    FROM Backend_Sales_OBT
    GROUP BY user_id
)
SELECT
    f.user_id,
    f.experiment_group,
    f.total_page_view,
    CASE WHEN b.user_id IS NOT NULL THEN 1 ELSE 0 END AS is_converted,
    COALESCE(b.total_spend, 0) AS total_spend

FROM frontend_agg AS f
LEFT JOIN backend_agg AS b 
    ON f.user_id = b.user_id
ORDER BY user_id
""")

┌─────────┬──────────────────┬─────────────────┬──────────────┬────────────────────┐
│ user_id │ experiment_group │ total_page_view │ is_converted │    total_spend     │
│  int64  │     varchar      │      int64      │    int32     │       double       │
├─────────┼──────────────────┼─────────────────┼──────────────┼────────────────────┤
│       0 │ NULL             │         1125645 │            0 │                0.0 │
│       2 │ Search           │              10 │            0 │                0.0 │
│       3 │ Organic          │               5 │            0 │                0.0 │
│       7 │ Email            │              10 │            0 │                0.0 │
│       8 │ Search           │              10 │            0 │                0.0 │
│       9 │ Search           │              14 │            1 │               28.0 │
│      10 │ Search           │               5 │            0 │                0.0 │
│      11 │ Search           │               5 │            0 │  

##### 5.5 Mart_Demographic_Preferences

In [30]:
duckdb.sql("""
SELECT
    CASE 
        WHEN age < 20 THEN 'Under 20'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        WHEN age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60 and Over' 
    END AS age_group,
    gender,
    product_category,
    SUM(sale_price) AS total_revenue,
    COUNT(DISTINCT user_id) AS unique_buyers,
    COUNT(order_items_id) AS items_sold
FROM Backend_Sales_OBT
WHERE items_status NOT IN ('Returned', 'Cancelled')
GROUP BY 1, 2, 3
ORDER BY total_revenue DESC
""")

┌─────────────┬─────────┬───────────────────────────────┬────────────────────┬───────────────┬────────────┐
│  age_group  │ gender  │       product_category        │   total_revenue    │ unique_buyers │ items_sold │
│   varchar   │ varchar │            varchar            │       double       │     int64     │   int64    │
├─────────────┼─────────┼───────────────────────────────┼────────────────────┼───────────────┼────────────┤
│ 60 and Over │ M       │ Intimates                     │ 16094.350017547607 │           200 │        230 │
│ 60 and Over │ M       │ Tops & Tees                   │ 14162.850003480911 │           189 │        210 │
│ 60 and Over │ M       │ Sleep & Lounge                │ 13974.630032539368 │           164 │        191 │
│ 60 and Over │ F       │ Intimates                     │ 13527.660044670105 │           202 │        231 │
│ 60 and Over │ M       │ Fashion Hoodies & Sweatshirts │ 12897.250036239624 │           171 │        185 │
│ 50-59       │ M       │ In

#### 5.6 Mart_Geo_LTV

In [31]:
duckdb.sql("""
SELECT
    user_id,
    ANY_VALUE(latitude) AS latitude,
    ANY_VALUE(longitude) AS longitude,
    ANY_VALUE(country) AS country,
    SUM(sale_price) AS lifetime_value,
    COUNT(DISTINCT order_id) AS total_orders
FROM Backend_Sales_OBT
WHERE items_status NOT IN ('Returned', 'Cancelled')
GROUP BY user_id
""")

┌─────────┬─────────────┬─────────────┬─────────┬────────────────────┬──────────────┐
│ user_id │  latitude   │  longitude  │ country │   lifetime_value   │ total_orders │
│  int64  │   double    │   double    │ varchar │       double       │    int64     │
├─────────┼─────────────┼─────────────┼─────────┼────────────────────┼──────────────┤
│   87954 │ 31.11201703 │ 121.3655885 │ China   │ 10.649999618530272 │            1 │
│   65619 │ 31.26245654 │ 121.3998975 │ China   │ 156.98000144958496 │            2 │
│   39173 │ 31.26245654 │ 121.3998975 │ China   │  95.98999786376952 │            1 │
│    5283 │ 31.21817204 │ 121.4113304 │ China   │  223.9900016784668 │            3 │
│    2072 │ 31.10174682 │ 121.4227589 │ China   │               45.0 │            1 │
│   70896 │ 31.41748495 │ 121.4243812 │ China   │ 197.98999977111816 │            1 │
│   21729 │ 31.27375242 │ 121.4341896 │ China   │  361.4499969482422 │            1 │
│   96695 │ 31.30871091 │ 121.4570442 │ China   │     

#### 5.7 Mart_Impulse_Penalty

In [ ]:
Mart_Session_Intent = duckdb.sql("""
WITH session_aggregated AS (
    SELECT 
        session_id,
        MAX(user_id) AS user_id, 
        DATE_DIFF('minute', MIN(created_at), MAX(created_at)) AS session_duration_minutes,
        COUNT(id) AS total_page_views,
        SUM(CASE WHEN event_type IN ('product', 'cart', 'department') THEN 1 ELSE 0 END) AS high_intent_clicks,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS is_purchase_session
    FROM events
    GROUP BY session_id
)
SELECT * 
FROM session_aggregated 
WHERE is_purchase_session = 1
""")

In [34]:
Mart_Impulse_Penalty = duckdb.sql("""
WITH intent_tagged AS (
    SELECT 
        session_id,
        user_id,
        session_duration_minutes,
        CASE 
            WHEN session_duration_minutes < 10 AND total_page_views <= 6 THEN 'Impulse Buyers'
            WHEN session_duration_minutes > 60 OR total_page_views >= 12 THEN 'Heavy Browsers'
            ELSE 'Targeted Searchers'
        END AS user_intent
    FROM Mart_Session_Intent
),
session_with_orders AS (
    SELECT 
        i.user_intent,
        i.session_duration_minutes,
        CASE WHEN o.items_status = 'Returned' THEN 1.0 ELSE 0.0 END AS is_returned
    FROM intent_tagged i
    JOIN Backend_Sales_OBT o ON i.user_id = o.user_id 
)

SELECT 
    user_intent AS User_Intent,
    AVG(is_returned) AS Return_Rate,
    AVG(CASE WHEN session_duration_minutes < 120 THEN session_duration_minutes ELSE 120 END) AS Avg_Session_Mins,
    COUNT(*) AS volume
FROM session_with_orders
GROUP BY user_intent
ORDER BY Return_Rate DESC
""")
Mart_Impulse_Penalty

┌────────────────────┬─────────────────────┬────────────────────┬────────┐
│    User_Intent     │     Return_Rate     │  Avg_Session_Mins  │ volume │
│      varchar       │       double        │       double       │ int64  │
├────────────────────┼─────────────────────┼────────────────────┼────────┤
│ Impulse Buyers     │ 0.15773740197220282 │  5.878600823045267 │  25758 │
│ Targeted Searchers │ 0.15446565921636052 │  10.53307392996109 │  11051 │
│ Heavy Browsers     │ 0.15281236172446402 │ 113.52624090092748 │ 115259 │
└────────────────────┴─────────────────────┴────────────────────┴────────┘